# MaleCNS architecture audit

Inspect the recorded full-data audit and its exact implementation. Counts refer to the non-null-superclass selection, not biological completeness. See `docs/DATASET_ARCHITECTURE_AUDIT.md` for interpretation and limitations.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
while not (root / "scripts/audit_malecns.py").exists():
    if root.parent == root:
        raise RuntimeError("Open this notebook from inside the FlyLab repository")
    root = root.parent
result = json.loads((root / "docs/results/malecns-architecture-audit.json").read_text())
print({k: result[k] for k in ["dataset", "generated_at", "included_neurons", "retained_connections", "retained_synapses", "hex_column_neurons"]})

{'dataset': 'male-cns:v1.0', 'generated_at': '2026-09-12T19:04:48.196765+00:00', 'included_neurons': 166700, 'retained_connections': 25582938, 'retained_synapses': 124177617, 'hex_column_neurons': 23720}


## Audit implementation

The following cell displays the actual code used for the streaming boundary scan, identity/hash checks, reciprocal-edge lookup and anatomical summaries. It reads local source data without modifying simulation state.

In [2]:
print((root / "scripts/audit_malecns.py").read_text())

"""Read-only anatomical/wiring audit; writes only the requested aggregate report.

Run: PYTHONPATH=backend uv run python scripts/audit_malecns.py
"""
from __future__ import annotations

import argparse
from collections import Counter
import csv
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.feather as feather


def digest(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def lookup(sorted_ids, values):
    """Indices in sorted unique IDs, with -1 for unknown segment IDs."""
    if not len(sorted_ids):
        return np.full(len(values), -1, dtype=np.int64)
    pos = np.searchsorted(sorted_ids, values)
    safe = np.minimum(pos, len(sorted_ids) - 1)
    return np.where((pos < len(sorted_ids)) & (sorted_ids[safe] == values), pos, -1)


def tally(codes, weights, size):
    """Count rows and sum weights in int64, without floating-p

## Accounting and feedback

These checks inspect the saved aggregate result. They do not rerun the raw-data scan. Traversal ranks summarize sensory-seeded graph traversal; they are not anatomical laminae.

In [3]:
blocks = result["superclass_connections"]
assert sum(x["connection_rows"] for x in blocks) == result["retained_connections"]
assert sum(x["synapses"] for x in blocks) == result["retained_synapses"]
assert sum(x["connections"] for x in result["traversal_direction"].values()) == result["retained_connections"]
assert result["directed_connections_with_reverse"] % 2 == 0
print("Reciprocal neuron pairs:", result["directed_connections_with_reverse"] // 2)
print(json.dumps(result["traversal_direction"], indent=2))

Reciprocal neuron pairs: 3823760
{
  "forward": {
    "connections": 8033503,
    "synapses": 42623322
  },
  "same_rank": {
    "connections": 12501804,
    "synapses": 59901012
  },
  "backward": {
    "connections": 5046987,
    "synapses": 21652035
  },
  "missing_rank": {
    "connections": 644,
    "synapses": 1248
  }
}


## Inspect the anatomical inventory

Literal labels remain unchanged, including composite types and curator status labels. The saved result contains the complete nonzero superclass connection matrix. No type-to-physiology mapping is inferred here.

In [4]:
print("Column-assigned neurons by type:", result["hex_type_neurons"])
print("Excluded annotation statuses:", result["excluded_status_counts"])
print("DN/AN neurons joined:", result["dnan_joined_neurons"])
print("DN/AN neurons with function labels:", result["dnan_with_function_annotation"])

Column-assigned neurons by type: {'L2': 1767, 'L1': 1767, 'Tm2': 1758, 'Mi9': 1760, 'C3': 1770, 'Mi1': 1762, 'T1': 1764, 'Mi4': 1758, 'Tm1': 1767, 'L5': 1773, 'Tm9': 1743, 'Tm20': 1732, 'L3': 892, 'Tm4': 833, 'C2': 874}
Excluded annotation statuses: {'Unimportant': 10751, 'Glia': 11864, 'Out of scope': 2594, 'Orphan': 12071, 'Orphan-artifact': 2292, 'Orphan hotknife': 1427, 'PRT Orphan': 76, 'Leaves': 416, '(missing)': 876, '0.5assign': 1832, 'Soma Anchor': 193, 'Prelim Roughly traced': 17, 'Anchor': 271, 'Roughly traced': 1, 'Hard to trace': 103, 'Partially traced': 9, 'Sensory Anchor': 78, 'RT Hard to trace': 2, 'Reviewed': 4}
DN/AN neurons joined: 3160
DN/AN neurons with function labels: 320


## Reproduce from source files

From the repository root, run:

```bash
PYTHONPATH=backend uv run python scripts/audit_malecns.py
PYTHONPATH=backend uv run pytest tests/test_dataset_audit.py -q
```

Requires the three local source Feather files, `data/full/` import and `data/anatomy-reference/` supplements. Exact public URLs and hashes are recorded in the result JSON (`sources` and `supplement.files`); save supplements by their `name` and place `supplement` in `data/anatomy-reference/sources.json`. The audit rejects hash mismatches. No downloads run when opening this notebook.

99% remains a proposed structural target; inspect identity preservation, directed-edge precision/recall, integer count errors and small populations separately. Physiology and behavior need independent validation.